# MLOps — Demo 1 · The manual way (the reproducibility and tracking problem)

**Live demo.** We train a model the way most people do — by hand, in one notebook.
Watch what we *lose* as we go.

Two rules for this demo:
1. We edit **one** training cell, in place, and re-run it. We never keep copies.
2. We never write anything down.

By the end, the model works — but nobody can say how it was made.

## 1. Load the data

In [1]:
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Real California housing data (ships with scikit-learn, downloads once)
data = fetch_california_housing(as_frame=True)

df = data.data.copy()
df["price"] = data.target * 100_000     # target is in $100k units -> real dollars

print(df.shape)
df.head()

(20640, 9)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,price
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,452600.0
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,358500.0
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,352100.0
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,341300.0
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,342200.0


## 2. The training cell

This is the **only** cell we change. Read the edit list at the top, then:

- **First:** run it **twice as-is** and watch the RMSE change. (No seed was set.)
- **Then:** make the edits one at a time, re-running after each. Only the latest survives.

In [ ]:
# ================== LIVE EDITS — make ONE at a time, re-run each ==================
# Run 0 : run this cell TWICE as-is  ->  RMSE changes      ->  "same result tomorrow?"
# Run 1 : n_estimators = 100  ->  300                      ->  "which run was best?"
# Run 2 : uncomment the rooms_per_person feature line      ->  "how was it made?"
# Run 3 : add  max_depth=15  to the model                  ->  "which settings gave that?"
# Run 4 : uncomment the data-cleaning line (drop capped)   ->  "which data was it trained on?"
#
# We never set a random seed, and never record any of this.   <- that's the whole point
# =================================================================================

d = df.copy()

# --- Run 4 edit: data cleaning (uncomment this line) ---
# d = d[d["price"] < 500_000]        # drop the price-capped rows

# --- Run 2 edit: feature engineering (uncomment this line) ---
# d["rooms_per_person"] = d["AveRooms"] / d["AveOccup"]

feature_cols = [c for c in d.columns if c != "price"]
X = d[feature_cols]
y = d["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# --- Run 1 edit: change 100 -> 300 --- | --- Run 3 edit: add max_depth=15 ---
model = RandomForestRegressor(n_estimators=100)
model.fit(X_train, y_train)

preds = model.predict(X_test)
rmse = mean_squared_error(y_test, preds) ** 0.5
print(f"RMSE: ${rmse:,.0f}")

## 3. The question

You just watched five different models go by. Quick — **without scrolling up**:

- What was the RMSE on **Run 2**?
- Was `max_depth` still set when you dropped the capped rows?
- Which run was actually the **best**, and what exactly produced it?

The model works. Everything needed to trust it, rebuild it, or improve it — is gone.
That is the **reproducibility and tracking** gap. Next: we fix it with MLflow.